# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [13]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame
from time import perf_counter
np.set_printoptions(precision=3, suppress=True)


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [ ]:
# PM - Point Marker
N_TOTAL_POINTS_PM = 20 # Desired total number of marker points
N_SURFACE_POINTS_PM = N_TOTAL_POINTS_PM - 4 # Amount needed to be generated after starting with a tetrahedron 
N_MAX_DIM_SIZE_PM = 75 # size parameter used to define the vertice.
OCCLUSION_THRESHOLD_PM = 4 # For this idea we wanted to decrease chance that points are not blocked when viewing from front in +z

# CM - Calibration Marker
N_TOTAL_POINTS_CM = 20
N_SURFACE_POINTS_CM = N_TOTAL_POINTS_CM - 4 # Amount points for generation
N_MAX_DIM_SIZE_CM = 50 # size parameter
OCCLUSION_THRESHOLD_CM = 4 # For this idea we wanted to make sure that points are not blocked when viewing from front in +z

N_OBSERVATIONS = 25  

OUTSIDE_CALIBRATION_TIP = [-10000, -10000, -10000]

N_RANDOM_TEST_POSES = 1000
    
# Multiple random seeds so no coupling of randomness between marker generation, tilt generation, simulation, and test generation randomness
SIMULATION_RANDOM_SEED = 42
BARYCENTRIC_RANDOM_SEED = 60
ANGLE_SIGN_RANDOM_SEED = 100
TEST_RANDOM_SEED = 500

# We set the random seed to ensure reproducibility in testing.
np.random.seed(SIMULATION_RANDOM_SEED)
# This random seed is so that there is no coupling of randomness between marker generation and simulation randomness
rng = np.random.default_rng(seed=BARYCENTRIC_RANDOM_SEED)
# This is a method to generate points on the barycentric faces of our normal/irregular tetrahedral marker bodies. This is used to generate the surface points of the pointer and calibration marker bodies, which are then used to generate the observations for the pointer tip calibration problem.
# The arguments are as follows:
# tetrahedral_positions: a list of 4 vct3 points that define the vertices of the tetrahedron
# threshold: a distance threshold to ensure that the generated points are not too close, this is minimum norm using the 3D distance between points
# occlusion_threshold: a distance threshold to ensure that the generated points are not too close in xy plane, which we believe would help with occlusion
# n_surface_points: the number of points to generate on the surface of the tetrahedron
def generate_points_on_surface(tetrahedral_positions, threshold, occlusion_threshold, n_surface_points):
    if n_surface_points <= 0:
        return list(tetrahedral_positions)
    selected_points = [point.vec.flatten() for point in tetrahedral_positions] 
    vertices = np.array(selected_points) 
    barycenteric_faces = [[0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3]] # the faces of the tetrahedron defined by the vertices
    max_uv, min_uv = 1, 0 # maximum and minimum weights for the vertices which we use to generate points on the faces of the tetrahedron
    discretization_step = 0.1 # the step size for discretizing the uv space, which we use to generate points on the faces of the tetrahedron
    uv_range = np.arange(min_uv, max_uv + discretization_step, discretization_step) # create steps to get evenly spaced regions to randomly sample from, generating the weights of vertices
    index_ranges = [(i, j) for i in range(len(uv_range) - 1) for j in range(len(uv_range) - 1)] # index based list to create all possible combinations for u,v weight generation 
    for i, j in rng.permutation(index_ranges): # mix up the order of index ranges to get a more random selection of points on the surface of the tetrahedron before early stopping
        u = rng.uniform(uv_range[i], uv_range[i + 1]) # for example if the uv step at i is 0.1 and at i+1 is 0.2, then we will randomly sample a value between 0.1 and 0.2 for u
        v = rng.uniform(uv_range[j], uv_range[j + 1]) 
        if u + v > 1: # weights (1-u-v, u, v) without negative weights must add up to 1, so if they don't, we flip them to be within the range of the triangle defined by the face of the tetrahedron
            u, v = 1 - u, 1 - v
        for face in rng.permutation(barycenteric_faces): # go through each face and see if we can generate a point on that surface, randomly mix order of faces so that we don't always generate points on the same face first
            point = ((1 - u - v) * vertices[face[0]] + u * vertices[face[1]] + v * vertices[face[2]]) # barycentric point definition through vertices and weights
            if all(np.linalg.norm(np.array(selected_point) - point) > threshold and np.linalg.norm(np.array(selected_point[:2]) - point[:2]) > occlusion_threshold for selected_point in selected_points):
                selected_points.append(point) # check if the point is far away by threshold and occlusion threshold 
            if len(selected_points) - len(tetrahedral_positions) >= n_surface_points:
                        break
        if len(selected_points) - len(tetrahedral_positions) >= n_surface_points:
            break
    return [vct3(point) for point in selected_points]


ptr_local_positions = [vct3(0, 0, N_MAX_DIM_SIZE_PM/2), vct3(N_MAX_DIM_SIZE_PM, 0, 0), vct3(0, N_MAX_DIM_SIZE_PM, N_MAX_DIM_SIZE_PM), vct3(N_MAX_DIM_SIZE_PM, N_MAX_DIM_SIZE_PM, N_MAX_DIM_SIZE_PM)] # tetrahedron  
threshold = N_MAX_DIM_SIZE_PM * 0.05 - 1 # Minimum distance between generated points, used this threshold since we do not want to overconstrain but also ensure spaced

ptr_local_positions = generate_points_on_surface(ptr_local_positions, threshold, OCCLUSION_THRESHOLD_PM, N_SURFACE_POINTS_PM) # converting the points back into vct3 objs, in the pointer frame
ptr_tip_approx = vct3(N_MAX_DIM_SIZE_PM/2, N_MAX_DIM_SIZE_PM/2, -N_MAX_DIM_SIZE_PM*(2/3))  

print("Amount of markers", len(ptr_local_positions)) # Validation for num of markers generated

Amount of markers 20


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [15]:
threshold = N_MAX_DIM_SIZE_CM * 0.05 - 1 # Minimum distance between generated points, used this threshold since we do not want to overconstrain but also ensure spaced

cal_local_positions = [
    vct3(0, N_MAX_DIM_SIZE_CM, 0),
    vct3(N_MAX_DIM_SIZE_CM, N_MAX_DIM_SIZE_CM, 0),
    vct3(N_MAX_DIM_SIZE_CM, 0, 0),
    vct3(N_MAX_DIM_SIZE_CM/2, N_MAX_DIM_SIZE_CM/2, N_MAX_DIM_SIZE_CM/2)
] # tetrahedron 


cal_local_positions = generate_points_on_surface(cal_local_positions, threshold, OCCLUSION_THRESHOLD_CM, N_SURFACE_POINTS_CM)

print("Amount of markers for calibration object ", len(cal_local_positions)) # Validation for num of markers generated

Amount of markers for calibration object  20


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [16]:
F_tracker_approx = Frame(Rot(axis='z', angle=0.5),vct3(300, 300, -1250))  # tracker pose relative to the workspace
F_cal_approx = Frame(Rot(axis='z', angle=0.5),vct3(300, 300, 250))      # calibration object pose relative to the workspace, within the tracker camera's field of view
# diff of 1350 mm within 1000 mm to 2000 mm range

## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [17]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions) #create the marker body in simulation for the pointer using the generated points
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions) #create the marker body in simulation for the calibration object using the generated points

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

# defining point and tracker in sim
ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb) 
trk = simulation.create_tracker("trk1", F_tracker_approx) 

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [18]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    """
    Using averaged obs to refine local marker pos
    input:
        trk - Tracker to read with
        placed - PlacedBody whose body's nominal marker positions should be
            refined in place (via body.set_marker_positions(...))
        n_readings - int, readings to average per pass
        passes - int, how many times to repeat
    output: None (mutates placed.body's nominal_marker_positions in place)
    """
    if n_readings < 1 or passes < 1:
        raise ValueError("We need sensor readings and finite amount of passes to do refinement")
    for pass_num in range(passes):
        # there are multiple readings for a single marker, so we will sum them up and then average them to get a more accurate reading of the marker position
        sum_point = {} 
        for read in range(n_readings): # camera readings 
            reading = (simulation.sample_marker_bodies(trk, [placed])[placed.body.name]) # dict with the body name as key and the reading as value
            reading_frame, reading_markers = reading.frame, reading.markers # get the pose relative to tracker and marker positions in tracker coordinates
            trk_mb = reading_frame.inv() 
            for i, marker in enumerate(reading_markers):
                if i not in sum_point:
                    sum_point[i] = vct3(0, 0, 0) 
                trk_marker = trk_mb*(marker.p) # convert the marker position from tracker frame to marker body frame
                sum_point[i] = sum_point[i] + trk_marker # sum up the marker positions for marker i in the marker body frame
        average_reading = []
        for x in range(len(sum_point)):
            avg_reading = sum_point[x] /n_readings # average positions for marker x 
            average_reading.append(avg_reading)
        placed.body.set_marker_positions(average_reading) # set the marker positions of the placed body to the average reading in the marker body frame
    return None


t_before_refining = perf_counter()
# refine marker body using tracker and placed body
refine_marker_body(trk, ptr_placed) 
refine_marker_body(trk, cal_placed) 

t_elapsed_refine_mb = perf_counter() - t_before_refining # calculting time it takes to refine mb

## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [19]:

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
sign_rng = np.random.default_rng(seed=ANGLE_SIGN_RANDOM_SEED)
t_before_pivot_calibration_obs = perf_counter()
for j in range(N_OBSERVATIONS):
    sign = sign_rng.choice([-1, 1])
    R_desired = Rot(axis='x', angle=np.pi/2 * sign * j / N_OBSERVATIONS) if j%2 == 0 else Rot(axis='y', angle=np.pi/2 * sign * j  / N_OBSERVATIONS) 
    # We chose this rotation method as it evenly samples the space of rotation with some randomness, additionally from pi/2 to -pi/2 covers the space tested
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    sensed = simulation.sense_tip(ptr, hg)

    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)
print(f"collected {len(F_ptr_list)} observations")
pivot_observation_time = perf_counter() - t_before_pivot_calibration_obs # calculating time it takes for pivot calibration

# Validation
assert len(F_ptr_list) == len(F_cal_list) == len(sensed_tip_list)


collected 25 observations


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

  Each observation `j` gives you three measurements of the same physical\n",
    "moment: `F_ptr,j` and `F_cal,j` from the tracker, and `p_sensed,j` from the\n",
    "calibration sensor. Derive how they relate to the unknown `p_tip`, and use\n",
    "all three in your calibration.\n",
    "\n",
    "**Corner case to consider**: What are the corner cases you want to analyse here? (e.g. what happens to observations whose `p_sensed,j` is `None`?)

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [20]:
def compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list):
    """
    input:
        F_ptr_list - List[Frame], the pointer marker body's pose relative to
            the tracker for each observation j (Fptr,j)
        F_cal_list - List[Frame], the calibration object's pose relative to
            the tracker for each observation j (Fcal,j), same order/length
        sensed_tip_list - List[Optional[vct3]], the calibration sensor's
            reading of the tip position relative to the calibration object
            for each observation j (p_sensed,j), same order/length; an entry
            is None if that observation's tip was out of sensor range
    output:
        p_tip - vct3, the calibrated tip position in the pointer's local frame
        cov_estimate - np.ndarray (3x3), an estimated covariance for p_tip
            (e.g. from the least-squares fit's residuals)
    """
    if not(len(sensed_tip_list)==len(F_cal_list)==len(F_ptr_list)):
        raise ValueError("Invalid input length")
    estimated_tips = []
    for index in range(len(sensed_tip_list)):
        sensed_tip = sensed_tip_list[index]
        F_ptr_trk = F_ptr_list[index] # ptr to trk coord
        F_cal_trk = F_cal_list[index] # cal to trk coord
        if sensed_tip is None or np.array_equal(sensed_tip.vec.flatten(),OUTSIDE_CALIBRATION_TIP):
            continue
        G_j = F_cal_trk.inv()*F_ptr_trk # ptr to cal coord
        p_tip_j = G_j.inv()*sensed_tip
        estimated_tips.append(p_tip_j.vec.flatten())
        

    if len(estimated_tips) < 2: raise ValueError("Covariance cannot be calc as insufficient estimate count") 

    estimated_tips_arr = np.asarray(estimated_tips)
    u_tip = np.mean(estimated_tips_arr,axis=0)
    cov = np.cov(estimated_tips_arr, rowvar=False, ddof=1)/len(estimated_tips) # estimated covariance of average tip pos in pointer coordinates
    return vct3(u_tip), cov
    

t_before_pivot_cal_comp = perf_counter()
p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list)
compute_pivot_calibration_time = perf_counter() - t_before_pivot_cal_comp

In [21]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))

# Validation checks for correctness in covariance (size, finite)
assert np.all(np.isfinite(p_tip_est.vec))
assert p_tip_cov.shape == (3, 3)
assert np.all(np.isfinite(p_tip_cov))

calibrated p_tip: vct3(x=37.23, y=38.42, z=-49.58)
Internal Vector:
[[ 37.227]
 [ 38.415]
 [-49.578]]
estimated covariance (mm^2):
 [[ 0.002 -0.    -0.   ]
 [-0.     0.002 -0.   ]
 [-0.    -0.     0.001]]
estimated std-dev per axis (mm): [0.041 0.041 0.028]


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [22]:
near_local_positions = []
N_SIZE = 200
for x in [-N_SIZE/2, N_SIZE/2]:
    for y in [-N_SIZE, N_SIZE]:
        for z in [-N_SIZE, N_SIZE]:
            near_local_positions.append(vct3(x, y, z))

# we made a rectanglular prism of eight points that is near the test pose which allows the ability to filter any noise generated from the tracker if the tracker itself shakes by comparing movement of reference with test pose movement

F_near_approx = Frame(Rot(),vct3(650, 300, 300))  # TODO: Frame, placed close to (not inside) the test-pose cube

t_before_near_refine = perf_counter()
near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)

t_elapsed_refine_near = perf_counter() - t_before_near_refine

Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [23]:
thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]
# TODO: the full grid above is large (7^3 * 5^3 = 42,875 combinations) --
# choose a representative subset of (position, orientation) test poses
# rather than every combination.

test_poses = [] # will build a list of test poses Frame(Rot.xyz(rx, ry, rz), vct3(300+tx, 300+ty, 300+tz))

minimal_offsets = [0, 200, -200]
test_thetas = thetas # we manipulated this if we wanted a smaller range of angles

for tx in minimal_offsets:
    for ty in minimal_offsets:
        for tz in minimal_offsets:
            position = vct3(300 + tx, 300 + ty, 300 + tz)
            test_poses.append(Frame(Rot(), position))
            for direction in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]:
                for theta in test_thetas:
                    if theta == 0:
                        continue  # Upright pose already included
                    rx = direction[0] * theta
                    ry = direction[1] * theta
                    rz = direction[2] * theta
                    test_poses.append(Frame(Rot.xyz(rx, ry, rz), position))
test_rng = np.random.default_rng(TEST_RANDOM_SEED)
for _ in range(N_RANDOM_TEST_POSES):
    rx, ry, rz = test_rng.choice(thetas, 3)
    random_rotation = Rot.xyz(rx,ry,rz)
    offset_x, offset_y, offset_z = test_rng.choice(offsets,3)
    random_offset = vct3(300 + offset_x, 300 + offset_y, 300 + offset_z)
    Frame_to_add = Frame(random_rotation, random_offset)
    test_poses.append(Frame_to_add)
# Above generates defining tests to ensure all thetas are used and then random combination of offsets and thetas test poses


test_results = []  # list of (F_t_nominal, error_mm)
t_before_testing = perf_counter()
for F_t_nominal in test_poses:
    test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise") # accurate placement of the pointer marker body at the nominal test pose
    obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed]) # observation of placement with presumed error of the poiner marker body and reference body
    p_tip_pred = obs[near_mb.name].frame.inv() * obs[ptr_mb.name].frame * p_tip_est # predicted tip of observed tracker frame with tracker jiggle removed using reference frame
    p_tip_true = near_placed.actual_placement.inv() * test_placed.actual_placement * ptr.get_actual_tip_position().p # tip position in reference body coordinates
    test_results.append((F_t_nominal, (p_tip_true - p_tip_pred).norm())) # error calculated as euclidean distance between tips
    # TODO, for each F_t_nominal:
    #   1. test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    #   2. obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])  -- single reading
    #   3. predict the tip position relative to near_placed using p_tip_est
    #   4. compute the TRUE tip position relative to near_placed using
    #      test_placed.actual_placement / near_placed.actual_placement (ground truth,
    #      for reporting only -- see markdown above)
    #   5. record the error and append (F_t_nominal, error) to test_results

t_elapsed_testing = perf_counter() - t_before_testing

## Achieved performance

Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [24]:
errors = np.array([error for _, error in test_results])
t_elapsed_calibration = pivot_observation_time + t_elapsed_refine_mb + compute_pivot_calibration_time
print(f"Test poses: {len(errors)}")
print(f"Mean error: {np.mean(errors):.3f} mm")
print(f"Median error: {np.median(errors):.3f} mm")
print(f"Maximum error: {np.max(errors):.3f} mm")
print(f"95th percentile error: {np.percentile(errors, 95):.3f} mm")
print(f"Within 2 mm: {np.sum(errors <= 2)} / {len(errors)} "
      f"({100 * np.mean(errors <= 2):.1f}%)")
print(f"RMSE = {np.sqrt(np.mean(errors**2)):.3f} mm")

print("\nCalibrated p_tip:", p_tip_est)
with np.printoptions(precision=6, suppress=False):
    print("Estimated tip covariance in pointer coordinates (mm^2):")
    print(p_tip_cov)
    print("Estimated standard deviation per axis (mm):")
    print(np.sqrt(np.diag(p_tip_cov)))

print("\nMeasured simulation runtime:")
print(f"Pointer and calibration marker refinement: {t_elapsed_refine_mb:.6f} s")
print(f"Calibration observation collection: {pivot_observation_time:.6f} s")
print(f"Tip calibration calculation: {compute_pivot_calibration_time:.6f} s")
print(f"Combined calibration runtime: {t_elapsed_calibration:.6f} s")
print(f"Reference marker refinement: {t_elapsed_refine_near:.6f} s")
print(f"Test loop runtime: {t_elapsed_testing:.6f} s")
print(f"Average test evaluation runtime: {t_elapsed_testing / len(errors):.6f} s")

Test poses: 1513
Mean error: 0.828 mm
Median error: 0.821 mm
Maximum error: 1.555 mm
95th percentile error: 1.330 mm
Within 2 mm: 1513 / 1513 (100.0%)
RMSE = 0.880 mm

Calibrated p_tip: vct3(x=37.23, y=38.42, z=-49.58)
Internal Vector:
[[ 37.227]
 [ 38.415]
 [-49.578]]
Estimated tip covariance in pointer coordinates (mm^2):
[[ 1.651260e-03 -2.489419e-05 -8.686938e-05]
 [-2.489419e-05  1.685631e-03 -2.126020e-04]
 [-8.686938e-05 -2.126020e-04  7.740353e-04]]
Estimated standard deviation per axis (mm):
[0.040636 0.041056 0.027821]

Measured simulation runtime:
Pointer and calibration marker refinement: 0.592112 s
Calibration observation collection: 0.177177 s
Tip calibration calculation: 0.007445 s
Combined calibration runtime: 0.776735 s
Reference marker refinement: 0.131934 s
Test loop runtime: 7.058090 s
Average test evaluation runtime: 0.004665 s
